# Olympic marathon with `gpc` + `pods`

CIP-001D Phase E. Modelling uses core `gpc` only; the dataset comes from external [`pods`](https://pypi.org/project/pods/).

```bash
pip install pods matplotlib   # or: pip install 'gpc[examples]'
```

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "gpc").is_dir():
    sys.path.insert(0, str(root))
elif (root.parent / "gpc").is_dir():
    sys.path.insert(0, str(root.parent))

import numpy as np
import gpc

try:
    import pods
except ImportError as exc:
    raise SystemExit("pip install pods  (or gpc[examples])") from exc

data = pods.datasets.olympic_marathon_men()
X = np.asarray(data["X"], dtype=float)
Y = np.asarray(data["Y"], dtype=float)
if X.ndim == 1:
    X = X[:, None]
if Y.ndim == 1:
    Y = Y[:, None]

m = gpc.GPRegression(X, Y)
m.set_by_path("rbf.lengthscale", 30.0)
m.optimize(iters=100)
print("objective:", m.objective())
print({p: m.get_by_path(p) for p in m.param_paths()})

In [ ]:
from gpc.plot import plot_1d
import matplotlib.pyplot as plt

plot_1d(m, X, Y)
plt.xlabel("year")
plt.ylabel("time (hours)")
plt.show()

Sparse DTCVAR variant (optional): place inducing inputs on a year grid.

In [ ]:
Z = np.linspace(float(X.min()), float(X.max()), 8)[:, None]
ms = gpc.SparseGPRegression(X, Y, Z)  # approx='dtcvar' default
ms.optimize(iters=80)
print("sparse objective:", ms.objective(), "approx:", ms.approx)
plot_1d(ms, X, Y, label="dtcvar mean")
plt.show()